In [1]:
pip install torch numpy tslearn pandas mantis-tsfm


Note: you may need to restart the kernel to use updated packages.


In [2]:
from sklearn.metrics import accuracy_score
from tslearn.shapelets import LearningShapelets
from tslearn.preprocessing import TimeSeriesScalerMeanVariance
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tslearn.datasets import UCR_UEA_datasets

In [3]:
from tslearn.datasets import UCR_UEA_datasets
# Load the LSST dataset from UEA archive
ds = UCR_UEA_datasets()
X_train, y_train, X_test, y_test = ds.load_dataset("LSST")

## Baseline model

In [4]:
shapelet_clf = LearningShapelets(
    n_shapelets_per_size={11: 50, 21: 50, 31: 50},  # More shapelets and varied lengths
    optimizer="adam",  # Use Adam optimizer
    max_iter=200,      # More iterations
    #verbose=1,
    random_state=42,
)
scaler = TimeSeriesScalerMeanVariance()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Fit the model
shapelet_clf.fit(X_train_scaled, y_train)

# Predict on the test set
y_pred = shapelet_clf.predict(X_test_scaled)
ytrain_pred = shapelet_clf.predict(X_train_scaled)

# Evaluate accuracy
accuracy = accuracy_score(y_train, ytrain_pred)
print(f"Shapelet Train Accuracy: {accuracy:.4f}")
accuracy = accuracy_score(y_test, y_pred)
print(f"Shapelet Test Accuracy: {accuracy:.4f}")

/opt/python/lib/python3.13/site-packages/tslearn/shapelets/shapelets.py:498: FutureWarning: The default value for 'scale' is set to False in version 0.4 to ensure backward compatibility, but is likely to change in a future version.
  warnings.warn("The default value for 'scale' is set to False "
2026-03-08 09:12:35.872460: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


Shapelet Train Accuracy: 0.6950
Shapelet Test Accuracy: 0.5203


## Step 2

In [7]:
from mantis.architecture import MantisV1
from mantis.trainer import MantisTrainer
from sklearn.preprocessing import LabelEncoder


In [5]:
def resize(X):
    X_scaled = F.interpolate(torch.tensor(X, dtype=torch.float), size=512, mode='linear', align_corners=False)
    return X_scaled.numpy()

In [8]:
X_train_resized = resize(X_train)
X_test_resized = resize(X_test)

le = LabelEncoder()
y_train_encoded = le.fit_transform(y_train)
y_test_encoded = le.transform(y_test)

In [ ]:
# Load the pre-trained Mantis model
network = MantisV1(device='cuda')
network = network.from_pretrained("paris-noah/Mantis-8M")

# Initialize the MantisTrainer
model = MantisTrainer(device='cuda', network=network)

# Fit the model using the encoded labels
model.fit(X_train_resized, y_train_encoded, batch_size=32)

# Predict probabilities and class labels
probs = model.predict_proba(X_test_resized)
y_pred_encoded = model.predict(X_test_resized)

# Decode the predicted labels back to original class names
y_pred = le.inverse_transform( y_pred_encoded)

Epoch 208: Train Loss 0.0000:  42%|████▏     | 209/500 [4:50:15<6:43:11, 83.13s/it]

In [ ]:
from mantis.adapters import MultichannelProjector

adapter = MultichannelProjector(new_num_channels=5, base_projector='pca')
adapter.fit(X_train_resized)
X_transformed = adapter.transform(X_train_resized)

model = MantisTrainer(device='cuda', network=network)
Z = model.transform(X_transformed)

